# Loremaster Generation Evaluation

This notebook runs two generation evals comparing `best_fields` (BM25 only, pre-hybrid) against `rrf_elser` (current production) across 10 curated benchmark questions.

## Evals

### 1. Generation Eval (`run_eval.py`)
Measures information quality — how good is the answer regardless of how it sounds.

| Dimension | Description | Direction |
|---|---|---|
| `correctness` | Factually accurate compared to the reference answer | Higher is better |
| `faithfulness` | Stayed within what was retrieved, no invented details | Higher is better |
| `relevance` | Actually answered the question asked | Higher is better |

Judged by Claude Haiku against a reference answer and the retrieved source snippets.

### 2. Embellishment Eval (`run_embellishment_eval.py`)
Measures how much narrative voice the model adds beyond what the source material contains.

| Field | Description |
|---|---|
| `source_score` | Narrative quality of the raw retrieved snippets (1=dry, 5=fully narrative) |
| `response_score` | Narrative quality of the model's response (1=dry, 5=fully narrative) |
| `delta` | `response_score - source_score` — how much narrative the model added |

Both scores use the same `score_narrative` function so they are directly comparable. The delta is computed arithmetically, not by the judge.

## Benchmark

10 questions across three categories: `lore` (6), `gameplay` (2), `item` (2). See `evals/generation/benchmark.json`.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path('../..').resolve()))

import anthropic
from elasticsearch import Elasticsearch
from config import ANTHROPIC_API_KEY, ES_ENDPOINT, ES_API_KEY, ES_INDEX
from evals.retrieval.strategies import STRATEGIES
from evals.generation.judge import score, score_narrative
from evals.generation.run_eval import search_with_snippets, generate

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
es     = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY)

BENCHMARK          = Path('../../evals/generation/benchmark.json')
DIMENSIONS         = ['correctness', 'faithfulness', 'relevance']
COMPARE_STRATEGIES = ['best_fields', 'rrf_elser']

## Load Benchmark

In [ ]:
benchmark  = json.loads(BENCHMARK.read_text())
categories = sorted(set(b['category'] for b in benchmark))

print(f'Total questions: {len(benchmark)}')
for cat in categories:
    print(f'  {cat}: {sum(1 for b in benchmark if b["category"] == cat)}')

## Run Generation Eval

For each strategy: retrieves snippets from ES, generates a response with Claude Haiku, then scores correctness, faithfulness, and relevance against the reference answer and snippets.

In [ ]:
gen_results = {}

for strategy_name in COMPARE_STRATEGIES:
    print(f'Running: {strategy_name}')
    results = []
    for entry in benchmark:
        snippets = search_with_snippets(entry['question'], strategy_name)
        response = generate(entry['question'], snippets)
        judgment = score(entry['question'], entry['reference'], snippets, response, client)
        results.append({**entry, 'response': response, 'snippets': snippets, **judgment})
    gen_results[strategy_name] = results
    print(f'  Done ({len(results)} questions scored)')

## Generation Results — Overall

In [ ]:
col = 14
print(f"{'Dimension':<15}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print(f'  {"delta":>8}')
print('-' * (15 + (col + 2) * len(COMPARE_STRATEGIES) + 10))

for dim in DIMENSIONS:
    avgs  = {n: sum(r[dim] for r in gen_results[n]) / len(gen_results[n]) for n in COMPARE_STRATEGIES}
    delta = avgs[COMPARE_STRATEGIES[-1]] - avgs[COMPARE_STRATEGIES[0]]
    print(f'{dim:<15}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print(f'  {delta:>+8.2f}')

## Generation Results — Per Category

In [ ]:
for category in categories:
    print(f'\n{category}')
    for dim in DIMENSIONS:
        avgs = []
        print(f'  {dim:<15}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in gen_results[name] if r['category'] == category]
            avg = sum(r[dim] for r in cat_results) / len(cat_results)
            avgs.append(avg)
            print(f'  {avg:>12.2f}', end='')
        print(f'  {avgs[-1] - avgs[0]:>+8.2f}')

## Run Embellishment Eval

Scores the narrative quality of the source snippets and the response independently using the same `score_narrative` function, then computes the delta. A large positive delta means the model added significant narrative voice beyond what the source contained.

In [ ]:
emb_results = {}

for strategy_name in COMPARE_STRATEGIES:
    print(f'Running: {strategy_name}')
    results = []
    for entry in benchmark:
        snippets = search_with_snippets(entry['question'], strategy_name)
        response = generate(entry['question'], snippets)

        source_j   = score_narrative('\n---\n'.join(snippets), client)
        response_j = score_narrative(response, client)

        results.append({
            **entry,
            'snippets':        snippets,
            'response':        response,
            'source_score':    source_j['narrative_score'],
            'response_score':  response_j['narrative_score'],
            'delta':           response_j['narrative_score'] - source_j['narrative_score'],
            'source_reasoning':   source_j['reasoning'],
            'response_reasoning': response_j['reasoning'],
        })
    emb_results[strategy_name] = results
    print(f'  Done ({len(results)} questions scored)')

## Embellishment Results — Overall

In [ ]:
col = 14
print(f"{'':18}", end='')
for name in COMPARE_STRATEGIES:
    print(f'  {name:>{col}}', end='')
print()
print('-' * (18 + (col + 2) * len(COMPARE_STRATEGIES)))

for field in ['source_score', 'response_score', 'delta']:
    avgs = {n: sum(r[field] for r in emb_results[n]) / len(emb_results[n]) for n in COMPARE_STRATEGIES}
    print(f'{field:<18}', end='')
    for name in COMPARE_STRATEGIES:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print()

## Embellishment Results — Per Category

In [ ]:
for category in categories:
    print(f'\n{category}')
    for field in ['source_score', 'response_score', 'delta']:
        print(f'  {field:<18}', end='')
        for name in COMPARE_STRATEGIES:
            cat_results = [r for r in emb_results[name] if r['category'] == category]
            avg = sum(r[field] for r in cat_results) / len(cat_results)
            print(f'  {avg:>12.2f}', end='')
        print()

## High Embellishment Flags

Questions where the model added 2 or more points of narrative beyond the source. These are the responses most likely to be adding voice the source does not support.

In [ ]:
for name in COMPARE_STRATEGIES:
    flagged = [r for r in emb_results[name] if r['delta'] >= 2]
    print(f'{name} — {len(flagged)} flagged')
    for r in flagged:
        print(f"  [{r['category']}] {r['question']}")
        print(f"    source={r['source_score']}  response={r['response_score']}  delta={r['delta']:+d}")
        print(f"    {r['response_reasoning']}")
    print()